# Sales Performance Analytics - validated workflow

**Business goal:** describe sales performance across time, customer segment,
geography, product category, and shipping mode for the US retail dataset.

This notebook separates line items from orders, validates the source before
analysis, and avoids profitability, ROI, and conversion claims that the
available fields cannot support.

## 1. Load the registered source table

Expected source: a Databricks table named `train` containing 9,800 sales
line items. Loading a table does not change it.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

sales_raw = spark.table("train")

print(f"Source rows: {sales_raw.count():,}")
display(sales_raw.limit(10))

Source rows: 9,800


Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96
2,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94
3,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62
4,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
5,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368
6,CA-2015-115812,2015-06-09,2015-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86
7,CA-2015-115812,2015-06-09,2015-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28
8,CA-2015-115812,2015-06-09,2015-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152
9,CA-2015-115812,2015-06-09,2015-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504
10,CA-2015-115812,2015-06-09,2015-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9


## 2. Validate schema and source assumptions

In [0]:
required_columns = {
    "Row ID", "Order ID", "Order Date", "Ship Date", "Ship Mode",
    "Customer ID", "Customer Name", "Segment", "Country", "City",
    "State", "Postal Code", "Region", "Product ID", "Category",
    "Sub-Category", "Product Name", "Sales"
}

missing_required_columns = sorted(required_columns - set(sales_raw.columns))
assert not missing_required_columns, (
    f"Missing required columns: {missing_required_columns}"
)

sales_raw.printSchema()

root
 |-- Row ID: long (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: long (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)



## 3. Prepare analysis fields

Identifier fields that are not needed for the present analysis are removed
from the presentation dataset. No rows are removed. Stable customer and
product identifiers should be retained in a future dimensional model.

In [0]:
sales_clean = (
    sales_raw
    .drop("Row ID", "Product ID", "Postal Code", "Customer ID")
    .withColumn("Order Date", F.to_date(F.col("Order Date")))
    .withColumn("Ship Date", F.to_date(F.col("Ship Date")))
    .withColumn("Sales", F.col("Sales").cast(DoubleType()))
    .withColumn("Order Year", F.year(F.col("Order Date")))
    .withColumn("Order Month", F.date_format(F.col("Order Date"), "yyyy-MM"))
)

source_rows = sales_raw.count()
clean_rows = sales_clean.count()
assert source_rows == clean_rows, (
    f"Unexpected row-count change: source={source_rows}, clean={clean_rows}"
)

print(f"Prepared rows: {clean_rows:,}")
display(sales_clean.limit(10))

Prepared rows: 9,800


Order ID,Order Date,Ship Date,Ship Mode,Customer Name,Segment,Country,City,State,Region,Category,Sub-Category,Product Name,Sales,Order Year,Order Month
CA-2017-152156,2017-11-08,2017-11-11,Second Class,Claire Gute,Consumer,United States,Henderson,Kentucky,South,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2017,2017-11
CA-2017-152156,2017-11-08,2017-11-11,Second Class,Claire Gute,Consumer,United States,Henderson,Kentucky,South,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,2017,2017-11
CA-2017-138688,2017-06-12,2017-06-16,Second Class,Darrin Van Huff,Corporate,United States,Los Angeles,California,West,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2017,2017-06
US-2016-108966,2016-10-11,2016-10-18,Standard Class,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,South,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,2016,2016-10
US-2016-108966,2016-10-11,2016-10-18,Standard Class,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,South,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2016,2016-10
CA-2015-115812,2015-06-09,2015-06-14,Standard Class,Brosina Hoffman,Consumer,United States,Los Angeles,California,West,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,2015,2015-06
CA-2015-115812,2015-06-09,2015-06-14,Standard Class,Brosina Hoffman,Consumer,United States,Los Angeles,California,West,Office Supplies,Art,Newell 322,7.28,2015,2015-06
CA-2015-115812,2015-06-09,2015-06-14,Standard Class,Brosina Hoffman,Consumer,United States,Los Angeles,California,West,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,2015,2015-06
CA-2015-115812,2015-06-09,2015-06-14,Standard Class,Brosina Hoffman,Consumer,United States,Los Angeles,California,West,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,2015,2015-06
CA-2015-115812,2015-06-09,2015-06-14,Standard Class,Brosina Hoffman,Consumer,United States,Los Angeles,California,West,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,2015,2015-06


## 4. Data-quality checks

An order may legitimately contain several product lines. Therefore repeated
Order IDs are not duplicates. Exact-row duplicates are reported for review
and are not removed automatically.

In [0]:
null_counts = sales_clean.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in sales_clean.columns
])

exact_duplicate_rows = clean_rows - sales_clean.dropDuplicates().count()
non_positive_sales = sales_clean.filter(F.col("Sales") <= 0).count()
invalid_ship_dates = sales_clean.filter(
    F.col("Ship Date") < F.col("Order Date")
).count()

quality_summary = sales_clean.agg(
    F.countDistinct("Order ID").alias("distinct_orders"),
    F.countDistinct("Customer Name").alias("distinct_customers"),
    F.min("Order Date").alias("first_order_date"),
    F.max("Order Date").alias("last_order_date"),
    F.round(F.sum("Sales"), 2).alias("total_sales")
).first().asDict()

validation_results = {
    "source_rows": source_rows,
    "prepared_rows": clean_rows,
    "exact_duplicate_rows_for_review": exact_duplicate_rows,
    "non_positive_sales_rows": non_positive_sales,
    "ship_date_before_order_date_rows": invalid_ship_dates,
    **quality_summary,
}

print(validation_results)
display(null_counts)

{'source_rows': 9800, 'prepared_rows': 9800, 'exact_duplicate_rows_for_review': 1, 'non_positive_sales_rows': 0, 'ship_date_before_order_date_rows': 0, 'distinct_orders': 4922, 'distinct_customers': 793, 'first_order_date': datetime.date(2015, 1, 3), 'last_order_date': datetime.date(2018, 12, 30), 'total_sales': 2261536.78}


Order ID,Order Date,Ship Date,Ship Mode,Customer Name,Segment,Country,City,State,Region,Category,Sub-Category,Product Name,Sales,Order Year,Order Month
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 5. Create the temporary analytical view

In [0]:
sales_clean.createOrReplaceTempView("sales_data")

## 6. Headline KPIs

`line_items` counts product rows. `orders` counts distinct Order IDs.

In [0]:
%sql
SELECT
  COUNT(*) AS line_items,
  COUNT(DISTINCT `Order ID`) AS orders,
  ROUND(SUM(Sales), 2) AS total_sales,
  ROUND(SUM(Sales) / COUNT(DISTINCT `Order ID`), 2) AS sales_per_order
FROM sales_data

line_items,orders,total_sales,sales_per_order
9800,4922,2261536.78,459.48


## 7. Sales by region

In [0]:
%sql
SELECT Region, ROUND(SUM(Sales), 2) AS total_sales
FROM sales_data
GROUP BY Region
ORDER BY total_sales DESC

Region,total_sales
West,710219.68
East,669518.73
Central,492646.91
South,389151.46


## 8. Sales by category and sub-category

In [0]:
%sql
SELECT Category, ROUND(SUM(Sales), 2) AS total_sales
FROM sales_data
GROUP BY Category
ORDER BY total_sales DESC

Category,total_sales
Technology,827455.87
Furniture,728658.58
Office Supplies,705422.33


In [0]:
%sql
SELECT `Sub-Category`, ROUND(SUM(Sales), 2) AS total_sales
FROM sales_data
GROUP BY `Sub-Category`
ORDER BY total_sales DESC

Sub-Category,total_sales
Phones,327782.45
Chairs,322822.73
Storage,219343.39
Tables,202810.63
Binders,200028.79
Machines,189238.63
Accessories,164186.7
Copiers,146248.09
Bookcases,113813.2
Appliances,104618.4


## 9. Chronological monthly sales

Year-month is retained so this is a continuous trend, not an aggregation of
all Januaries, Februaries, and so on.

In [0]:
%sql
SELECT `Order Month` AS month, ROUND(SUM(Sales), 2) AS total_sales
FROM sales_data
GROUP BY `Order Month`
ORDER BY month

month,total_sales
2015-01,14205.71
2015-02,4519.89
2015-03,55205.8
2015-04,27906.85
2015-05,23644.3
2015-06,34322.94
2015-07,33781.54
2015-08,27117.54
2015-09,81623.53
2015-10,31453.39


## 10. Customers and states by recorded sales

In [0]:
%sql
SELECT `Customer Name`, ROUND(SUM(Sales), 2) AS total_sales
FROM sales_data
GROUP BY `Customer Name`
ORDER BY total_sales DESC
LIMIT 10

Customer Name,total_sales
Sean Miller,25043.05
Tamara Chand,19052.22
Raymond Buch,15117.34
Tom Ashbrook,14595.62
Adrian Barton,14473.57
Ken Lonsdale,14175.23
Sanjit Chand,14142.33
Hunter Lopez,12873.3
Sanjit Engle,12209.44
Christopher Conant,12129.07


In [0]:
%sql
SELECT State, ROUND(SUM(Sales), 2) AS total_sales
FROM sales_data
GROUP BY State
ORDER BY total_sales DESC

State,total_sales
California,446306.46
New York,306361.15
Texas,168572.53
Washington,135206.85
Pennsylvania,116276.65
Florida,88436.53
Illinois,79236.52
Michigan,76136.07
Ohio,75130.35
Virginia,70636.72


## 11. Shipping mode by distinct orders

The original notebook used `COUNT(*)`, which counted line items while
labelling them as orders. This query uses distinct Order IDs.

In [0]:
%sql
SELECT
  `Ship Mode`,
  COUNT(DISTINCT `Order ID`) AS number_of_orders
FROM sales_data
GROUP BY `Ship Mode`
ORDER BY number_of_orders DESC

Ship Mode,number_of_orders
Standard Class,2945
Second Class,944
First Class,772
Same Day,261


## 12. Correct average order value by segment

First aggregate product lines into orders, then average the order totals.

In [0]:
%sql
WITH order_totals AS (
  SELECT
    Segment,
    `Order ID`,
    SUM(Sales) AS order_sales
  FROM sales_data
  GROUP BY Segment, `Order ID`
)
SELECT
  Segment,
  COUNT(*) AS number_of_orders,
  ROUND(AVG(order_sales), 2) AS average_order_value
FROM order_totals
GROUP BY Segment
ORDER BY average_order_value DESC

Segment,number_of_orders,average_order_value
Home Office,894,475.37
Corporate,1491,461.77
Consumer,2537,452.53


## 13. Monthly sales by segment

In [0]:
%sql
SELECT
  `Order Month` AS month,
  Segment,
  ROUND(SUM(Sales), 2) AS total_sales
FROM sales_data
GROUP BY `Order Month`, Segment
ORDER BY month, Segment

month,Segment,total_sales
2015-01,Consumer,6896.63
2015-01,Corporate,1701.53
2015-01,Home Office,5607.55
2015-02,Consumer,3167.85
2015-02,Corporate,1183.67
2015-02,Home Office,168.37
2015-03,Consumer,11869.3
2015-03,Corporate,10952.17
2015-03,Home Office,32384.32
2015-04,Consumer,9108.61


## 14. Complete export for Power BI

Do **not** download the visible Databricks result table: it may be truncated.
Write the dataframe programmatically and confirm the exported row count.
Change the path if an existing folder must be preserved.

In [0]:
catalog_name = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema_name = spark.sql(
    "SELECT current_schema()"
).first()[0]

table_name = "superstore_sales_clean_v2"

powerbi_sales = sales_clean.select(
    F.col("Order ID").alias("order_id"),
    F.col("Order Date").alias("order_date"),
    F.col("Ship Date").alias("ship_date"),
    F.col("Ship Mode").alias("ship_mode"),
    F.col("Customer Name").alias("customer_name"),
    F.col("Segment").alias("segment"),
    F.col("Country").alias("country"),
    F.col("City").alias("city"),
    F.col("State").alias("state"),
    F.col("Region").alias("region"),
    F.col("Category").alias("category"),
    F.col("Sub-Category").alias("sub_category"),
    F.col("Product Name").alias("product_name"),
    F.col("Sales").alias("sales"),
    F.col("Order Year").alias("order_year"),
    F.col("Order Month").alias("order_month"),
)

full_table_name = (
    f"{catalog_name}.{schema_name}.{table_name}"
)

if spark.catalog.tableExists(full_table_name):
    raise ValueError(
        f"Safety stop: {full_table_name} already exists. "
        "Choose a new versioned name instead of overwriting it."
    )

(
    powerbi_sales
    .write
    .format("delta")
    .saveAsTable(full_table_name)
)

published_table = spark.table(full_table_name)

published_summary = published_table.agg(
    F.count("*").alias("rows"),
    F.countDistinct("order_id").alias("orders"),
    F.round(F.sum("sales"), 2).alias("total_sales")
).first().asDict()

assert published_summary["rows"] == clean_rows, (
    f"Published row-count validation failed: "
    f"{published_summary}"
)

assert (
    published_summary["orders"]
    == quality_summary["distinct_orders"]
), (
    f"Published order-count validation failed: "
    f"{published_summary}"
)

assert (
    published_summary["total_sales"]
    == quality_summary["total_sales"]
), (
    f"Published sales validation failed: "
    f"{published_summary}"
)

print(f"Published table: {full_table_name}")
print(published_summary)

Published table: workspace.default.superstore_sales_clean_v2
{'rows': 9800, 'orders': 4922, 'total_sales': 2261536.78}


## 15. Interpretation boundaries

Supported conclusions describe recorded sales, order counts, customer and
product concentration, shipping usage, and time patterns.

This dataset does **not** contain profit, cost, margin, campaign spend,
impressions, conversion, inventory, or returns. It therefore cannot establish
profitability, marketing ROI, conversion performance, or causal impact.